# Start Partition Influence Comparison

This notebook analyzes how the choice of start partition affects the quality and runtime of the local-search pipeline.

It produces:

1. a comparison of relative solution quality and winner rate
2. a comparison of runtime

The analysis is grouped by graph type, size class, and density regime.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [2]:
PLATEAU_STEPS = 4
PLATEAU_PIPELINE = ",".join(
    ["move_plateau"] * PLATEAU_STEPS
)

GRAPH_ORDER = ["powerlaw", "er"]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

PREFERRED_START_PARTITION_ORDER = [
    "singleton",
    "matching",
    "maximum_matching",
    "maximum_matching_edge_cover",
    "high_degree_first_matching",
    "high_degree_product_matching",
    "leiden_mdgp",
    "kapoce",
]

RESULTS_DIR = Path("../../results/experiment2/zero_gain_limit")

RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"

## Load and verify experiment data

Before the analysis, the notebook lists the contained pipelines, start partitions, zero-gain factors, and number of runs. This provides a compact check that the intended experiment results were loaded.

In [3]:
raw_all = pd.read_csv(RAW_RESULTS_FILE)

required_columns = {
    "pipeline",
    "start_partition",
    "zero_gain_factor",
    "run",
    "graph_type",
    "size_class",
    "regime",
    "dataset",
    "instance",
    "final_density",
    "ls_runtime",
    "num_moves",
    "num_passes",
}

missing_columns = required_columns.difference(raw_all.columns)

if missing_columns:
    raise ValueError("Missing required columns: " + ", ".join(sorted(missing_columns)))

experiment_check = (
    raw_all
    .groupby(
        [
            "pipeline",
            "start_partition",
            "zero_gain_factor",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        num_runs=("run", "nunique"),
        num_instances=("instance", "nunique"),
        num_rows=("run", "size"),
    )
    .sort_values(
        [
            "pipeline",
            "start_partition",
            "zero_gain_factor",
        ],
        na_position="first",
    )
    .reset_index(drop=True)
)

print(f"Loaded {len(raw_all):,} rows from {RAW_RESULTS_FILE}")

experiment_check

Loaded 640,000 rows from ../../results/experiment2/zero_gain_limit/raw_results.csv


,pipeline,start_partition,zero_gain_factor,num_runs,num_instances,num_rows
0,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_first_matching,1,10,2000,20000
1,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_first_matching,2,10,2000,20000
2,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_first_matching,4,10,2000,20000
3,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_first_matching,8,10,2000,20000
4,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_product_matching,1,10,2000,20000
5,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_product_matching,2,10,2000,20000
6,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_product_matching,4,10,2000,20000
7,"move_plateau,move_plateau,move_plateau,move_pl...",high_degree_product_matching,8,10,2000,20000
8,"move_plateau,move_plateau,move_plateau,move_pl...",kapoce,1,10,2000,20000
9,"move_plateau,move_plateau,move_plateau,move_pl...",kapoce,2,10,2000,20000


## Prepare start-partition experiments

In [4]:
raw_all["dataset_group"] = (
        raw_all["size_class"].astype(str)
        + " "
        + raw_all["regime"].astype(str)
)

raw_all["graph_type"] = pd.Categorical(
    raw_all["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

raw_all["dataset_group"] = pd.Categorical(
    raw_all["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

start_partition_raw = raw_all[
    (raw_all["pipeline"] == PLATEAU_PIPELINE)
    & (raw_all["zero_gain_factor"] == 4)
].copy()

if start_partition_raw.empty:
    raise ValueError("No rows found for the expected start partition experiment configuration.")

present_start_partitions = (start_partition_raw["start_partition"].dropna().astype(str).unique().tolist())

START_PARTITION_ORDER = [
    start_partition
    for start_partition
    in PREFERRED_START_PARTITION_ORDER
    if start_partition in present_start_partitions
]

START_PARTITION_ORDER += sorted(set(present_start_partitions).difference(START_PARTITION_ORDER))

start_partition_raw["start_partition"] = (
    pd.Categorical(
        start_partition_raw["start_partition"],
        categories=START_PARTITION_ORDER,
        ordered=True,
    )
)

print("Compared start partitions:", ", ".join(START_PARTITION_ORDER))

Compared start partitions: singleton, matching, maximum_matching, maximum_matching_edge_cover, high_degree_first_matching, high_degree_product_matching, leiden_mdgp, kapoce


## Solution quality

For every instance and start partition, only the run with the highest final solution quality is retained. The best result across all compared start partitions is used as the reference.

Relative solution quality is defined as

$
\\frac{\\text{best result across all start partitions}}
     {\\text{result of the respective start partition}}.
$

A value of $1.0$ means that the start-partition matches the best result found on the same instance. Values greater than $1.0$ indicate the remaining quality gap.

In [5]:
best_run_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "start_partition",
]

best_runs = (
    start_partition_raw
    .sort_values(
        [
            "final_density",
            "ls_runtime",
            "run",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .groupby(
        best_run_keys,
        observed=True,
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)


In [6]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = best_runs.pivot_table(
    index=instance_keys,
    columns="start_partition",
    values="final_density",
    observed=True,
)

density_table = density_table[START_PARTITION_ORDER]
density_table.columns.name = "start_partition"

if density_table.isna().any().any():
    incomplete_instances = density_table[density_table.isna().any(axis=1)]
    raise ValueError(f"{len(incomplete_instances)} instances do not contain results for all start-partitions.")

In [7]:
best_per_instance = density_table.max(axis=1)

relative_to_best = density_table.rdiv(best_per_instance, axis=0)

relative_quality_summary = (
    relative_to_best
    .groupby(
        level=["graph_type", "dataset_group"]
    )
    .agg(["mean", "min", "max"])
    .stack(level=0, future_stack=True)
    .reset_index()
    .rename(
        columns={
            "mean": "mean_relative_to_best",
            "min": "min_relative_to_best",
            "max": "max_relative_to_best",
        }
    )
)

relative_quality_summary

,graph_type,dataset_group,start_partition,mean_relative_to_best,min_relative_to_best,max_relative_to_best
0,powerlaw,small sparse,singleton,1.003132,1.0,1.019452
1,powerlaw,small sparse,matching,1.002632,1.0,1.012658
2,powerlaw,small sparse,maximum_matching,1.001936,1.0,1.014369
3,powerlaw,small sparse,maximum_matching_edge_cover,1.002003,1.0,1.014971
4,powerlaw,small sparse,high_degree_first_matching,1.002690,1.0,1.012911
...,...,...,...,...,...,...
59,er,large dense,maximum_matching_edge_cover,1.010255,1.0,1.034436
60,er,large dense,high_degree_first_matching,1.010402,1.0,1.038733
61,er,large dense,high_degree_product_matching,1.010289,1.0,1.032169
62,er,large dense,leiden_mdgp,1.006800,1.0,1.027530


In [8]:
overall_relative_quality_summary = (
    relative_to_best
    .agg(["mean", "min", "max"])
    .T
    .reset_index()
    .rename(
        columns={
            "index": "start_partition",
            "mean": "mean_relative_to_best",
            "min": "min_relative_to_best",
            "max": "max_relative_to_best",
        }
    )
)

overall_relative_quality_summary

,start_partition,mean_relative_to_best,min_relative_to_best,max_relative_to_best
0,singleton,1.004715,1.0,1.034864
1,matching,1.004580,1.0,1.037656
2,maximum_matching,1.004227,1.0,1.036580
3,maximum_matching_edge_cover,1.004244,1.0,1.034436
4,high_degree_first_matching,1.004665,1.0,1.038733
5,high_degree_product_matching,1.004621,1.0,1.032169
6,leiden_mdgp,1.005928,1.0,1.038346
7,kapoce,1.006204,1.0,1.067798


## Runtime

The reported runtime is the total local-search runtime of all runs for one instance and operator, averaged over all instances in the corresponding dataset group.

In [9]:
runtime_per_instance = (
    start_partition_raw
    .groupby(
        best_run_keys,
        observed=True,
    )
    .agg(
        total_runtime=("ls_runtime", "sum"),
        moves_per_run=("num_moves", "mean"),
        passes_per_run=("num_passes", "mean"),
    )
    .reset_index()
)

runtime_summary = (
    runtime_per_instance
    .groupby(
        [
            "graph_type",
            "dataset_group",
            "start_partition",
        ],
        observed=True,
    )
    .agg(
        mean_runtime=("total_runtime", "mean"),
        mean_moves_per_run=("moves_per_run", "mean"),
        mean_passes_per_run=("passes_per_run", "mean"),
    )
    .reset_index()
)

runtime_summary

,graph_type,dataset_group,start_partition,mean_runtime,mean_moves_per_run,mean_passes_per_run
0,powerlaw,small sparse,singleton,15.788470,2511.4092,350.9308
1,powerlaw,small sparse,matching,15.669917,2452.9352,351.5456
2,powerlaw,small sparse,maximum_matching,15.527568,2421.9012,343.7900
3,powerlaw,small sparse,maximum_matching_edge_cover,15.642526,2421.6144,343.9092
4,powerlaw,small sparse,high_degree_first_matching,15.924232,2454.6364,354.5532
...,...,...,...,...,...,...
59,er,large dense,maximum_matching_edge_cover,412.109630,16367.5252,179.6740
60,er,large dense,high_degree_first_matching,398.694572,16382.8832,178.5088
61,er,large dense,high_degree_product_matching,403.425740,16382.8580,178.1784
62,er,large dense,leiden_mdgp,497.546552,16289.0748,211.5516


## Combined comparison data

In [10]:
comparison_summary = (
    relative_quality_summary
    .merge(
        runtime_summary,
        on=[
            "graph_type",
            "dataset_group",
            "start_partition",
        ],
    )
)

comparison_summary["start_partition"] = pd.Categorical(
    comparison_summary["start_partition"],
    categories=START_PARTITION_ORDER,
    ordered=True,
)

comparison_summary = (
    comparison_summary
    .sort_values(
        [
            "graph_type",
            "dataset_group",
            "start_partition",
        ]
    )
    .reset_index(drop=True)
)

comparison_summary

,graph_type,dataset_group,start_partition,mean_relative_to_best,min_relative_to_best,max_relative_to_best,mean_runtime,mean_moves_per_run,mean_passes_per_run
0,powerlaw,small sparse,singleton,1.003132,1.0,1.019452,15.788470,2511.4092,350.9308
1,powerlaw,small sparse,matching,1.002632,1.0,1.012658,15.669917,2452.9352,351.5456
2,powerlaw,small sparse,maximum_matching,1.001936,1.0,1.014369,15.527568,2421.9012,343.7900
3,powerlaw,small sparse,maximum_matching_edge_cover,1.002003,1.0,1.014971,15.642526,2421.6144,343.9092
4,powerlaw,small sparse,high_degree_first_matching,1.002690,1.0,1.012911,15.924232,2454.6364,354.5532
...,...,...,...,...,...,...,...,...,...
59,er,large dense,maximum_matching_edge_cover,1.010255,1.0,1.034436,412.109630,16367.5252,179.6740
60,er,large dense,high_degree_first_matching,1.010402,1.0,1.038733,398.694572,16382.8832,178.5088
61,er,large dense,high_degree_product_matching,1.010289,1.0,1.032169,403.425740,16382.8580,178.1784
62,er,large dense,leiden_mdgp,1.006800,1.0,1.027530,497.546552,16289.0748,211.5516


## Pairwise comparison with the selected start partition

The previous analyses compare all start partitions simultaneously. However, the goal of this experiment is to select a single start partition for the remaining experiments. Therefore, the following analysis directly compares the selected reference start partition (`maximum_matching`) with each alternative.

For each competing start partition, the table reports

- the fraction of instances on which the reference start partition produces a better solution,
- the fraction of instances on which the competing start partition performs better,
- the mean relative quality advantage of the reference start partition,
- the mean quality loss when the reference start partition is outperformed, and
- the difference in the mean local-search runtime.

This pairwise comparison provides a more intuitive assessment of whether the selected start partition consistently outperforms its competitors than the overall winner rate alone.

In [11]:
REFERENCE = "maximum_matching"

reference_runtime = (
    runtime_per_instance[runtime_per_instance["start_partition"] == REFERENCE]
    .set_index(instance_keys)["total_runtime"]
)

pairwise_rows = []

for other in START_PARTITION_ORDER:
    if other == REFERENCE:
        continue

    reference_quality = density_table[REFERENCE]
    other_quality = density_table[other]

    other_runtime = (
        runtime_per_instance[runtime_per_instance["start_partition"] == other]
        .set_index(instance_keys)["total_runtime"]
    )

    pairwise_rows.append(
        {
            "Opponent": other,
            "MM better rate": (reference_quality > other_quality).mean(),
            "MM worse rate": (reference_quality < other_quality).mean(),
            "Runtime difference (%)": (other_runtime.mean() - reference_runtime.mean()) / reference_runtime.mean() * 100,
        }
    )

pairwise_summary = (pd.DataFrame(pairwise_rows))

pairwise_summary["Opponent"] = pd.Categorical(
    pairwise_summary["Opponent"],
    categories=[
        p for p in PREFERRED_START_PARTITION_ORDER
        if p != REFERENCE
    ],
    ordered=True,
)

pairwise_summary = (
    pairwise_summary
    .sort_values("Opponent")
    .reset_index(drop=True)
)

pairwise_summary

,Opponent,MM better rate,MM worse rate,Runtime difference (%)
0,singleton,0.5525,0.3605,-3.783332
1,matching,0.5295,0.3770,-5.513374
2,maximum_matching_edge_cover,0.2100,0.2010,0.854656
3,high_degree_first_matching,0.5275,0.3735,-7.460101
4,high_degree_product_matching,0.5235,0.3755,-6.784904
5,leiden_mdgp,0.6040,0.3230,3.123139
6,kapoce,0.5410,0.4125,225.628603


## LaTeX helper functions

In [12]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor

def latex_start_partition(start_partition: str) -> str:
    return r"\texttt{" + start_partition.replace("_", r"\_") + "}"

def format_number(value: float, decimals: int) -> str:
    return f"{value:.{decimals}f}"

def format_percent(value: float, decimals: int = 1) -> str:
    return (rf"{truncate_number(100 * value, decimals):.{decimals}f}\,\%"
    )

def format_signed_number(value: float, decimals: int) -> str:
    return f"{value:+.{decimals}f}"

## Build quality LaTeX table

In [13]:
def make_quality_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    graph_df = df[df["graph_type"] == graph_type].copy()

    lines = [
        r"\begin{table}[p]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        (
            r"\begin{tabular}{lp{5.0cm}r}"
        ),
        r"\toprule",
        (
            r"Datensatz & Startpartition & \shortstack{Mittlere relative\\Lösungsqualität} \\"
        ),
        r"\midrule",
    ]

    for dataset_index, dataset in enumerate(DATASET_ORDER):
        part = graph_df[graph_df["dataset_group"] == dataset].copy()

        part["start_partition"] = pd.Categorical(
            part["start_partition"],
            categories=START_PARTITION_ORDER,
            ordered=True,
        )

        part = part.sort_values("start_partition")

        best_mean = part["mean_relative_to_best"].min()

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}"
                rf"{{{dataset}}}"
                if row_index == 0
                else ""
            )

            mean_quality = format_number(row.mean_relative_to_best, 6)

            if np.isclose(row.mean_relative_to_best, best_mean):
                mean_quality = (rf"\textbf{{{mean_quality}}}")

            lines.append(
                f"{dataset_cell} "
                f"& {latex_start_partition(str(row.start_partition))} "
                f"& {mean_quality} "
                r"\\"
            )

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(
                r"\cmidrule(l){1-3}"
            )

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [14]:
powerlaw_quality_latex = make_quality_latex_table(
    comparison_summary,
    graph_type="powerlaw",
    caption=(
        "Vergleich der Lösungsqualität verschiedener Startpartitionen auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung der jeweiligen Startpartition. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:start_partition_quality",
)

print(powerlaw_quality_latex)

\begin{table}[t]
\centering
\caption{Vergleich der Lösungsqualität verschiedener Startpartitionen auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung der jeweiligen Startpartition. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:start_partition_quality}
\begin{tabular}{lp{5.0cm}r}
\toprule
Datensatz & Startpartition & \shortstack{Mittlere relative\\Lösungsqualität} \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & 1.003132 \\
 & \texttt{matching} & 1.002632 \\
 & \texttt{maximum\_matching} & \textbf{1.001936} \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.002003 \\
 & \texttt{high\_degree\_first\_matching} & 1.002690 \\
 & \texttt{high\_degree\_product\_matching} & 1.002661 \\
 & \texttt{leiden\_mdgp} & 1.006005 \\
 & \texttt{kapoce} & 1.004331 \\
\cmidrule(l){1-3}
\multirow{8}{*}{small dense} & \texttt{singleton} & 1.004302 \\
 & \texttt{matching} & 1.004596 \\

In [15]:
er_quality_latex = make_quality_latex_table(
    comparison_summary,
    graph_type="er",
    caption=(
        "Vergleich der Lösungsqualität verschiedener Startpartitionen auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung der jeweiligen Startpartition. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:start_partition_quality",
)

print(er_quality_latex)

\begin{table}[t]
\centering
\caption{Vergleich der Lösungsqualität verschiedener Startpartitionen auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung der jeweiligen Startpartition. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:start_partition_quality}
\begin{tabular}{lp{5.0cm}r}
\toprule
Datensatz & Startpartition & \shortstack{Mittlere relative\\Lösungsqualität} \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & 1.004289 \\
 & \texttt{matching} & \textbf{1.003906} \\
 & \texttt{maximum\_matching} & 1.004661 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.004575 \\
 & \texttt{high\_degree\_first\_matching} & 1.004324 \\
 & \texttt{high\_degree\_product\_matching} & 1.004368 \\
 & \texttt{leiden\_mdgp} & 1.005637 \\
 & \texttt{kapoce} & 1.009435 \\
\cmidrule(l){1-3}
\multirow{8}{*}{small dense} & \texttt{singleton} & \textbf{1.004848} \\
 & \texttt{matching} & 1.

In [16]:
def make_pairwise_latex_table(pairwise_df: pd.DataFrame, quality_df: pd.DataFrame, caption: str, label: str) -> str:
    quality = (quality_df.set_index("start_partition")["mean_relative_to_best"])

    lines = [
        r"\begin{table}[H]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{p{4.8cm}rrrr}",
        r"\toprule",
        (
            r"Startpartition "
            r"& \shortstack{Mittlere relative\\Lösungsqualität} "
            r"& \shortstack{Maximum\\Matching\\besser} "
            r"& \shortstack{Maximum\\Matching\\schlechter} "
            r"& \shortstack{Laufzeit-\\differenz (\%)} \\"
        ),
        r"\midrule",
    ]

    for start_partition in PREFERRED_START_PARTITION_ORDER:

        quality_value = format_number(quality[start_partition], 6)

        if start_partition == "maximum_matching":
            quality_value = rf"\textbf{{{quality_value}}}"

            lines.append(
                f"{latex_start_partition(start_partition)} "
                f"& {quality_value} "
                f"& -- & -- & -- \\\\"
            )
            continue

        row = pairwise_df[pairwise_df["Opponent"] == start_partition].iloc[0]

        lines.append(
            f"{latex_start_partition(start_partition)} "
            f"& {quality_value} "
            f"& {format_percent(row['MM better rate'], 1)} "
            f"& {format_percent(row['MM worse rate'], 1)} "
            f"& {format_signed_number(row['Runtime difference (%)'], 1)}\,\% "
            r"\\"
        )

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [17]:
pairwise_latex = make_pairwise_latex_table(
    pairwise_summary, overall_relative_quality_summary,
    caption=(
        "Paarweiser Vergleich von \\texttt{maximum\\_matching} mit den alternativen Startpartitionen über alle Instanzen. Die Laufzeitdifferenz gibt die prozentuale Änderung der mittleren Laufzeit gegenüber \\texttt{maximum\\_matching} an. Positive Werte entsprechen einer höheren, negative Werte einer geringeren Laufzeit der alternativen Startpartition."
    ),
    label="tab:maximum_matching_pairwise",
)

print(pairwise_latex)

\begin{table}[t]
\centering
\caption{Paarweiser Vergleich von \texttt{maximum\_matching} mit den alternativen Startpartitionen über alle Instanzen. Die Laufzeitdifferenz gibt die prozentuale Änderung der mittleren Laufzeit gegenüber \texttt{maximum\_matching} an. Positive Werte entsprechen einer höheren, negative Werte einer geringeren Laufzeit der alternativen Startpartition.}
\label{tab:maximum_matching_pairwise}
\begin{tabular}{p{5.35cm}rrrr}
\toprule
Startpartition & \shortstack{Mittlere relative\\Lösungsqualität} & \shortstack{Maximum\\Matching\\besser} & \shortstack{Maximum\\Matching\\schlechter} & \shortstack{Laufzeit-\\differenz (\%)} \\
\midrule
\texttt{singleton} & 1.004715 & 55.2\,\% & 36.0\,\% & -3.8\,\% \\
\texttt{matching} & 1.004580 & 52.9\,\% & 37.7\,\% & -5.5\,\% \\
\texttt{maximum\_matching} & \textbf{1.004227} & -- & -- & -- \\
\texttt{maximum\_matching\_edge\_cover} & 1.004244 & 21.0\,\% & 20.1\,\% & +0.9\,\% \\
\texttt{high\_degree\_first\_matching} & 1.004665 & 52